# section 3 — retrieval shortcut baselines (data-distribution problem)

See `README.md` in this folder for the full plan and reasoning. This section reproduces two
spectrum-blind baselines from MassSpecGym v1.5's pitfalls paper (`papers/MassSpecGym v1.5 common
pitfalls.pdf`, Appendix A.4, Methods 5-6) against the real held-out test fold — a checkpoint that
your evaluation harness scores correctly before exploring whether the underlying problem can be
reframed. See `REVIEW.md` for why these two baselines' exact scoring code isn't publicly available
anywhere in the MassSpecGym repo, and what that means for the choices you'll make below.

In [1]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from rdkit import Chem

from massspecgym.data import RetrievalDataset, MassSpecDataModule
from massspecgym.models.retrieval import RetrievalMassSpecGymModel

pl.seed_everything(0)

# Start small while the pipeline is still being debugged -- cheap to get wrong small, expensive
# to get wrong at full scale (782,695 unique candidate molecules in the real test fold).
# Set to None once both baselines are verified correct on the pilot slice.
PILOT_N_QUERIES = 50

CACHE_DIR = Path("pubchem_cache")
CACHE_DIR.mkdir(exist_ok=True)

Seed set to 0


## Phase 0 — load the real test fold

Unlike Section 2's 5-spectrum debug set, this section evaluates against MassSpecGym's actual
held-out test fold. Build a `RetrievalDataset` with `candidates_pth='standard'` (the default
mass-based retrieval candidates) and no `pth=` override, so it downloads the real
`MassSpecGym1.5.tsv` + candidates from HuggingFace rather than Section 2's debug files.

No `mol_transform` is needed here (leave it at its default) -- these baselines score candidates
directly from their SMILES strings (RDKit atom properties / PubChem lookups), not from a learned
fingerprint space. With `mol_transform=None`, what ends up in `batch["candidates_mol"]` versus
`batch["candidates_smiles"]`? Check `RetrievalDataset.__getitem__` in your installed package
(`.venv/Lib/site-packages/massspecgym/data/datasets.py`) if it's not obvious which one you actually
want to use below.

Wire the dataset into a `MassSpecDataModule` -- you don't need `split_pth`, since the real
dataset's own `fold` column already has train/val/test labels. Call `.prepare_data()` then
`.setup(stage=...)` -- look up what the `stage` argument controls in
`massspecgym/data/data_module.py`, and why `"test"` is the right value here rather than the
`stage=None` Section 2 used.

Note: this downloads and holds the *entire* candidates JSON in memory (~7.3M unique molecules
across all folds, not just test). Expect this to take a bit and use noticeable RAM -- that's
normal, not a sign something broke.

In [ ]:
dataset = RetrievalDataset(
    candidates_pth='standard',
)

data_module = MassSpecDataModule(
    dataset=dataset,
    batch_size=None,  # pick something -- does batch size matter here the way it did in Section 2?
)
data_module.prepare_data()
data_module.setup(stage=None)  # is this the right stage for what we're about to do?

## Phase 1 — chiral-atom-count baseline (Method 6)

Fully specified in Appendix A.4: for each candidate molecule, count atoms `a` where
`a.GetChiralTag() != Chem.ChiralType.CHI_UNSPECIFIED` (parse each SMILES with
`Chem.MolFromSmiles` first). Rank candidates by that count.

The ranking *direction* is not fixed -- the paper decides it empirically from the training set:
descending if ground-truth molecules carry more annotated stereocenters on average than their
candidates, ascending otherwise. That means before you can score the test fold, you need to compute
this once from training-fold data: average chiral-atom count across training ground-truth query
molecules, versus average chiral-atom count across their candidates. `dataset.candidates` (the raw
dict loaded from the candidates JSON) and `dataset.metadata` (with its `fold`/`smiles` columns) have
what you need -- how you structure that one-time computation is your design choice.

Subclass `RetrievalMassSpecGymModel` the same way Section 2's model did, but with
`configure_optimizers` returning `None` (see `massspecgym/models/retrieval/random.py`'s
`RandomRetrieval` for the precedent -- no optimizer, and `step()` still needs to return a `loss` key
even though nothing will ever use it for a gradient step).

In [ ]:
class ChiralAtomCountRetrieval(RetrievalMassSpecGymModel):
    def __init__(self, direction: str, **kwargs):
        """
        Args:
            direction: "ascending" or "descending" -- decided empirically from the training set
                before this model is constructed (see markdown above).
        """
        super().__init__(**kwargs)
        self.direction = direction

    def step(self, batch: dict, stage=None) -> dict:
        pass

    def configure_optimizers(self):
        pass

## Phase 2 — PubChem default-ranking baseline (Method 5)

Appendix A.4 defines this as ranking candidates in descending order of "PubChem CID deposition
count" -- see `REVIEW.md` for why the exact field is a best-effort interpretation (number of
Substance IDs mapping to a CID) rather than a verified fact, since no reference code for this
baseline is publicly available.

The PubChem lookup itself is infrastructure -- provided below. It does two things per unique
SMILES: (1) resolve to a PubChem CID via PUG-REST, (2) count how many Substance IDs map to that
CID. Both steps batch multiple identifiers per request (PubChem's usage policy expects this, not
one-molecule-per-call) and cache results to `CACHE_DIR` so re-running this notebook doesn't re-hit
the network.

Before running this on the full pilot set, sanity-check `pubchem_deposition_counts([...])` on 2-3
molecules you already know the answer for -- e.g. glucose (very well-known, should score high) and
a SMILES string you don't expect PubChem to recognize well. If the ordering doesn't match your
intuition, the interpretation in `REVIEW.md` may need revisiting before you trust it at scale.

In [ ]:
import urllib.request
import urllib.error

PUBCHEM_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
BATCH_SIZE = 100
REQUEST_DELAY_S = 0.25  # ~4 req/s, under PubChem's ~5 req/s guidance


def _pug_get(url: str) -> dict:
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=30) as resp:
                return json.loads(resp.read())
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return {}
            time.sleep(2 ** attempt)
        except (urllib.error.URLError, TimeoutError):
            time.sleep(2 ** attempt)
    return {}


def _load_cache(name: str) -> dict:
    pth = CACHE_DIR / f"{name}.json"
    return json.loads(pth.read_text()) if pth.exists() else {}


def _save_cache(name: str, cache: dict) -> None:
    (CACHE_DIR / f"{name}.json").write_text(json.dumps(cache))


def smiles_to_cids(smiles_list: list[str]) -> dict[str, int | None]:
    """Resolve each SMILES to a PubChem CID (None if unresolved), batched and cached."""
    cache = _load_cache("smiles_to_cid")
    todo = [s for s in dict.fromkeys(smiles_list) if s not in cache]
    for i in range(0, len(todo), BATCH_SIZE):
        batch = todo[i : i + BATCH_SIZE]
        url = f"{PUBCHEM_BASE}/compound/smiles/cids/JSON"
        data = f"smiles={urllib.parse.quote(','.join(batch))}".encode()
        req = urllib.request.Request(url, data=data)
        try:
            with urllib.request.urlopen(req, timeout=30) as resp:
                result = json.loads(resp.read())
            props = result.get("IdentifierList", {}).get("CID", [])
            # NOTE: batched SMILES->CID does not reliably preserve input order/alignment for
            # invalid or duplicate-hitting entries. Verify this assumption on your pilot slice
            # before trusting it at full scale -- consider falling back to one-by-one lookups
            # (slower, but unambiguous) if the batch endpoint's alignment doesn't hold up.
            for smi, cid in zip(batch, props):
                cache[smi] = cid
        except (urllib.error.HTTPError, urllib.error.URLError):
            for smi in batch:
                cache[smi] = None
        time.sleep(REQUEST_DELAY_S)
    _save_cache("smiles_to_cid", cache)
    return {s: cache.get(s) for s in smiles_list}


def cid_sid_counts(cids: list[int]) -> dict[int, int]:
    """Number of Substance IDs mapping to each CID, batched and cached."""
    cache = _load_cache("cid_sid_count")
    todo = [c for c in dict.fromkeys(cids) if str(c) not in cache]
    for i in range(0, len(todo), BATCH_SIZE):
        batch = todo[i : i + BATCH_SIZE]
        ids = ",".join(str(c) for c in batch)
        url = f"{PUBCHEM_BASE}/compound/cid/{ids}/sids/JSON"
        result = _pug_get(url)
        for info in result.get("InformationList", {}).get("Information", []):
            cache[str(info["CID"])] = len(info.get("SID", []))
        time.sleep(REQUEST_DELAY_S)
    return {c: cache.get(str(c), 0) for c in cids}


def pubchem_deposition_counts(smiles_list: list[str]) -> dict[str, int]:
    """End-to-end: SMILES -> CID -> SID count, defaulting to 0 for anything unresolved."""
    cid_map = smiles_to_cids(smiles_list)
    valid_cids = [c for c in cid_map.values() if c is not None]
    sid_counts = cid_sid_counts(valid_cids)
    return {
        smi: sid_counts.get(cid, 0) if cid is not None else 0
        for smi, cid in cid_map.items()
    }

In [ ]:
# Sanity check before trusting this at scale -- does the ordering match intuition?
_sanity_smiles = [
    "OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O",  # glucose -- should score high
]
pubchem_deposition_counts(_sanity_smiles)

Now the part that isn't infrastructure: turn `pubchem_deposition_counts` into a `step()` on a
second `RetrievalMassSpecGymModel` subclass. Same shape as Phase 1's model, but the score per
candidate comes from this lookup instead of an RDKit atom count. Think about *when* you call
`pubchem_deposition_counts` -- once per batch inside `step()` (simple, but re-looks-up repeated
candidates across batches even with the on-disk cache dict-lookups), or precomputed once over the
full deduplicated candidate set before evaluation even starts (matches the "dedupe globally" hard
constraint in `README.md`).

In [ ]:
class PubChemDefaultRankingRetrieval(RetrievalMassSpecGymModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def step(self, batch: dict, stage=None) -> dict:
        pass

    def configure_optimizers(self):
        pass

## Phase 3 — run both baselines with `WandbLogger` + `trainer.test()`

No `.fit()` -- there's nothing to train. `RandomRetrieval`'s precedent (`configure_optimizers`
returns `None`) means calling `trainer.fit()` would be a well-defined no-op anyway, but
`trainer.test()` directly is the honest way to express "this model doesn't learn."

Run on `PILOT_N_QUERIES` first. Once both baselines look right on the pilot slice, bump it to the
full test fold and re-run.

In [ ]:
chiral_model = ChiralAtomCountRetrieval(direction="?")  # fill in from your Phase 1 computation
pubchem_model = PubChemDefaultRankingRetrieval()

for name, model in [("chiral", chiral_model), ("pubchem", pubchem_model)]:
    wandb_logger = WandbLogger(project="massspecgym-section03", name=name)
    trainer = pl.Trainer(logger=wandb_logger, accelerator="cpu")
    trainer.test(model, data_module)

## Phase 4 — compare against the paper

Reported in Table 3 (Hit rate@1, with 99.9% bootstrap CI): chiral-atom baseline 4.88%
(4.36-5.44%), PubChem default ranking 49.98% (48.72-51.20%). How close do your numbers land, and if
they diverge, what's your best explanation given what `REVIEW.md` says about the code not being
independently verifiable? Write the comparison up in `REVIEW.md`, not just here.

In [ ]:
pass